# API 2.1.4: an invoice, from Tool to Environment

LangGraph, Python and deterministic checks only. Import and execute in order. The offline guard prevents connections and application processes. Install the matching 2.1.4 SDK; live results belong to a separate report.

In [ ]:
import pydantic
from pathlib import Path
import sys
root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    if root == root.parent:
        raise RuntimeError('Open this notebook from the repository')
    root = root.parent
sys.path.insert(0, str(root / 'tutorials'))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
'A small, offline invoice: Tool → Agent → System → Environment, with Eval.'
import agentic_systems as toolkit

class Invoice(pydantic.BaseModel):
    subtotal: int
    total: int | None = None
    currency: str | None = None
    status: str | None = None

@toolkit.tool
def quote(subtotal: int, *, context: toolkit.Context) -> dict:
    """Calculate integer money using the domain's tax policy."""
    return {'total': subtotal * (100 + context.data['tax_percent']) // 100, 'currency': context.data['currency']}


## Context, Skills and Memory
Context fixes semantics; Skill declares available operations; Memory retains local episodes.

In [ ]:
domain = toolkit.Context({'tax_percent': 16, 'currency': 'MXN'}, sources=('tax-policy-v1',))
capability = toolkit.skill(name='invoicing', tools=(quote,), instructions='Amounts are integer MXN.')
memory = toolkit.Memory(capacity=3)


## Agent
**Input** subtotal=100 → **process** quote Tool applies tax from Context → **output** total=116 MXN.

In [ ]:
agent_graph = toolkit.Graph(Invoice)
agent_graph.add_node('quote', quote)
agent_graph.add_edge(toolkit.START, 'quote')
agent_graph.add_edge('quote', toolkit.END)
agent = toolkit.Agent('accountant', graph=agent_graph, skills=(capability,), context=domain, memory=memory)
single = agent.run({'subtotal': 100})
print(single.data)


## System and decision
The Agent is a node. The graph approves amounts up to 1000 and routes larger amounts to review. compile prepares; run executes.

In [ ]:
system_graph = toolkit.Graph(Invoice)
system_graph.add_node('accountant', agent)
def approve_step(state):
    return {'status': 'approved'}
system_graph.add_node('approve', approve_step)
def review_step(state):
    return {'status': 'review'}
system_graph.add_node('review', review_step)
system_graph.add_edge(toolkit.START, 'accountant')
system_graph.add_conditional_edges('accountant', lambda state: 'review' if state.total > 1000 else 'approve', ['approve', 'review'])
system_graph.add_edge('approve', toolkit.END)
system_graph.add_edge('review', toolkit.END)
system = toolkit.System('billing', graph=system_graph, context=domain)
team = system.compile().run({'subtotal': 100})
print(team.data)


## Environment
**Input** two records → **process** System per record and state update → **output** two steps, sum=348.

In [ ]:
environment_graph = toolkit.Graph(Invoice)
environment_graph.add_node('billing', system)
environment_graph.add_edge(toolkit.START, 'billing')
environment_graph.add_edge('billing', toolkit.END)
environment = toolkit.Environment('batch', graph=environment_graph, records=({'subtotal': 100}, {'subtotal': 200}), state={'sum': 0}, update=lambda state, record, result: {'sum': state['sum'] + result.data['total']})
batch = environment.run()
print(batch.data['summary'])


## Eval and lineage
Eval checks the System explicitly. Humans and agents read two views of the same result.

In [ ]:
evaluation = toolkit.Eval('invoice-contract', system, cases=({'input': {'subtotal': 100}, 'expected': {'ok': True}},), checks=(lambda result, case: {'ok': result.data['total'] == 116 and result.data['currency'] == 'MXN'},))
checked = evaluation.run()
assert single.ok and single.data['total'] == 116
assert team.ok and team.data['status'] == 'approved'
assert batch.ok and batch.data['summary']['state'] == {'sum': 348}
assert checked.ok
assert all((r.check_invariants().ok for r in (single, team, batch, checked)))
assert blocked_attempts() == 0
print('Eval:', checked.ok)
print(batch.lineage().render())
print('Local episodes:', len(memory.read()))
